# TUGAS MANDIRI — Pertemuan 6
## Pipeline ETL Produksi: Transaksi + Produk + Ulasan Pelanggan

| | |
|---|---|
| **Nama** | Hilmi Mufid |
| **NIM** | 2505060046 |
| **Program Studi** | S1 Teknologi Informasi — Universitas Tidar |
| **Mata Kuliah** | Praktikum Big Data |
| **Pertemuan** | 6 |

---

### Konteks

Tim data engineering platform e-commerce (skenario lanjutan dari Pertemuan 2-5) meminta dibangun **pipeline ETL produksi pertama** yang menggabungkan tiga sumber data sekaligus: transaksi (CSV), data produk (JSON), dan data ulasan pelanggan (CSV terpisah) — merepresentasikan kondisi nyata di mana data tersebar di berbagai sistem dengan format berbeda-beda.

Notebook ini membangun pipeline lengkap mengikuti struktur **Extract → Transform → Load**, ditutup dengan satu insight bisnis di bagian E.

> **Catatan lingkungan & path:** Dijalankan di laptop Ubuntu (dual-boot), Spark/PySpark 3.5.9, Hadoop 3.4.3. Mengikuti konvensi direktori kerja pribadi yang dipakai sejak Pertemuan 3 (`/home/mufid/praktikum-bigdata0046`), maka `/user/[username]/tugas6` pada instruksi soal disesuaikan menjadi `/home/mufid/praktikum-bigdata0046/tugas6`.

---
## Persiapan: Membuat Tiga Sumber Data & SparkSession

Sesuai instruksi modul, tiga sumber data disiapkan sekaligus: transaksi (CSV, 5000 baris), produk (JSON Lines, 30 baris), dan ulasan (CSV, 3500 baris — tidak semua transaksi memiliki ulasan, kondisi yang realistis).

In [1]:
import numpy as np
import pandas as pd
import json

np.random.seed(33)

# Sumber 1: Transaksi (CSV) — 5000 baris
n_trx = 5000
kategori_list = ["Elektronik", "Fashion", "Makanan", "Rumah Tangga", "Kesehatan"]
df_trx_tugas6 = pd.DataFrame({
    "order_id": [f"TX{i}" for i in range(n_trx)],
    "product_id": np.random.randint(1, 31, size=n_trx),
    "unit_terjual": np.random.randint(1, 8, size=n_trx),
    "tanggal": np.random.choice(pd.date_range("2026-10-01","2026-10-31"), size=n_trx).astype(str),
})
df_trx_tugas6.to_csv("tugas6_transaksi.csv", index=False)

# Sumber 2: Data produk (JSON Lines) — 30 produk
produk = [
    {"product_id": i, "nama_produk": f"Produk-{i}", "kategori": np.random.choice(kategori_list),
     "harga": int(np.random.choice([25000,50000,75000,100000,150000,250000]))}
    for i in range(1, 31)
]
with open("tugas6_produk.json", "w") as f:
    for p in produk:
        f.write(json.dumps(p) + "\n")

# Sumber 3: Data ulasan (CSV) — tidak semua transaksi memiliki ulasan (realistis)
n_review = 3500
df_review = pd.DataFrame({
    "order_id": np.random.choice(df_trx_tugas6["order_id"], size=n_review, replace=False),
    "rating": np.random.randint(1, 6, size=n_review),
})
df_review.to_csv("tugas6_ulasan.csv", index=False)

print(f"Tiga sumber data berhasil dibuat:")
print(f"- tugas6_transaksi.csv : {len(df_trx_tugas6)} baris")
print(f"- tugas6_produk.json   : {len(produk)} baris")
print(f"- tugas6_ulasan.csv    : {len(df_review)} baris (tidak seluruh transaksi memiliki ulasan)")

Tiga sumber data berhasil dibuat:
- tugas6_transaksi.csv : 5000 baris
- tugas6_produk.json   : 30 baris
- tugas6_ulasan.csv    : 3500 baris (tidak seluruh transaksi memiliki ulasan)


In [11]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, round as spark_round

spark = SparkSession.builder \
    .appName("Tugas6-PipelineETL") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9


---
## A. EXTRACT *(bobot 15%)*

**Penjelasan:** Ketiga sumber data dibaca menjadi tiga Spark DataFrame terpisah, sesuai format aslinya masing-masing — `spark.read.csv()` untuk kedua berkas CSV, dan `spark.read.json()` untuk berkas JSON Lines (persis pola yang dipakai pada Sub-bab 6.3 modul).

In [12]:
df_transaksi = spark.read.csv("tugas6_transaksi.csv", header=True, inferSchema=True)
df_produk = spark.read.json("tugas6_produk.json")
df_ulasan = spark.read.csv("tugas6_ulasan.csv", header=True, inferSchema=True)

print("=== df_transaksi ===")
print("Jumlah baris:", df_transaksi.count())
df_transaksi.printSchema()

print("=== df_produk ===")
print("Jumlah baris:", df_produk.count())
df_produk.printSchema()

print("=== df_ulasan ===")
print("Jumlah baris:", df_ulasan.count())
df_ulasan.printSchema()

=== df_transaksi ===
Jumlah baris: 5000
root
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)

=== df_produk ===
Jumlah baris: 30
root
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- product_id: long (nullable = true)

=== df_ulasan ===
Jumlah baris: 3500
root
 |-- order_id: string (nullable = true)
 |-- rating: integer (nullable = true)



---
## B. TRANSFORM — Penggabungan *(bobot 25%)*

**Penjelasan:** Dua jenis join dipilih sesuai karakteristik datanya masing-masing:
- **`left join`** dari transaksi ke ulasan — karena **tidak semua transaksi memiliki ulasan** (hanya 3500 dari 5000), `left join` memastikan seluruh transaksi tetap tampil, dengan `rating` bernilai `null` untuk yang belum diulas. Jika dipakai `inner join`, 1500 transaksi tanpa ulasan akan hilang begitu saja dari data — tidak sesuai kebutuhan analisis.
- **`inner join`** dari transaksi ke produk — karena **setiap transaksi pasti memiliki `product_id` yang valid** (data referensi produk lengkap, 30 produk mencakup seluruh `product_id` yang mungkin muncul di transaksi), sehingga `inner join` aman dipakai tanpa risiko kehilangan baris.

Kolom `total_pendapatan` dihitung sebagai `unit_terjual x harga`.

In [13]:
# Left join: transaksi -> ulasan (tidak semua transaksi memiliki ulasan)
df_gabung = df_transaksi.join(df_ulasan, on="order_id", how="left")

# Inner join: transaksi -> produk (setiap transaksi pasti punya produk yang valid)
df_gabung = df_gabung.join(df_produk, on="product_id", how="inner")

# Menambahkan kolom total_pendapatan
df_gabung = df_gabung.withColumn("total_pendapatan", col("unit_terjual") * col("harga"))

print("Jumlah baris setelah kedua join:", df_gabung.count())
df_gabung.select("order_id", "product_id", "nama_produk", "kategori", "rating", "total_pendapatan").show(10)

Jumlah baris setelah kedua join: 5000
+--------+----------+-----------+------------+------+----------------+
|order_id|product_id|nama_produk|    kategori|rating|total_pendapatan|
+--------+----------+-----------+------------+------+----------------+
|     TX0|        21|  Produk-21|  Elektronik|     1|           50000|
|     TX1|         8|   Produk-8|     Fashion|  NULL|         1500000|
|     TX2|        25|  Produk-25|  Elektronik|     2|          100000|
|     TX3|         3|   Produk-3|     Fashion|     5|          300000|
|     TX4|        19|  Produk-19|Rumah Tangga|  NULL|          450000|
|     TX5|        10|  Produk-10|  Elektronik|     5|          225000|
|     TX6|        26|  Produk-26|   Kesehatan|     4|          450000|
|     TX7|         4|   Produk-4|     Makanan|  NULL|          150000|
|     TX8|         7|   Produk-7|  Elektronik|     5|          150000|
|     TX9|        30|  Produk-30|  Elektronik|  NULL|          250000|
+--------+----------+-----------+------

---
## C. TRANSFORM — Penanganan Data Kosong & Pengayaan *(bobot 20%)*

**Penjelasan:** Kolom `ada_ulasan` **wajib dibuat sebelum** `rating` diisi dengan `na.fill()` — urutan ini penting, karena begitu `rating` sudah diisi 0, Spark tidak bisa lagi membedakan "transaksi yang memang mendapat rating 0" dengan "transaksi yang sebenarnya tidak memiliki ulasan sama sekali". Kolom `ada_ulasan` dibuat otomatis (bukan diisi manual satu-satu) menggunakan `when().otherwise()` yang mengecek `col("rating").isNotNull()`.

**Kenapa mengisi `rating` kosong dengan 0 (bukan nilai lain, misalnya rata-rata rating)?** Rating kosong di sini bukan berarti "pelanggan menilai produk dengan skor rendah", melainkan **"pelanggan belum memberi ulasan sama sekali"** — situasi yang secara semantik berbeda dari nilai rating asli (1-5). Mengisi dengan rata-rata rating justru akan menyamarkan/mengaburkan fakta bahwa transaksi tersebut sebenarnya tidak memiliki ulasan, dan bisa menyesatkan analisis rata-rata rating produk. Angka 0 dipilih sebagai penanda netral "tidak ada data", sejalan dengan kolom pendamping `ada_ulasan` yang secara eksplisit menandai status tersebut.

In [14]:
# Kolom ada_ulasan dibuat SEBELUM na.fill(), dari kondisi rating masih null atau tidak
df_gabung = df_gabung.withColumn(
    "ada_ulasan",
    when(col("rating").isNotNull(), True).otherwise(False)
)

# Verifikasi jumlah transaksi tanpa ulasan sebelum diisi
jumlah_tanpa_ulasan = df_gabung.filter(col("rating").isNull()).count()
print(f"Jumlah transaksi tanpa ulasan (rating kosong): {jumlah_tanpa_ulasan}")

# Mengisi rating kosong dengan 0 menggunakan na.fill()
df_gabung = df_gabung.na.fill({"rating": 0})

# Verifikasi tidak ada lagi nilai kosong pada rating
print("Jumlah rating kosong setelah ditangani:", df_gabung.filter(col("rating").isNull()).count())

df_gabung.select("order_id", "product_id", "rating", "ada_ulasan", "kategori", "total_pendapatan").show(10)

Jumlah transaksi tanpa ulasan (rating kosong): 1500
Jumlah rating kosong setelah ditangani: 0
+--------+----------+------+----------+------------+----------------+
|order_id|product_id|rating|ada_ulasan|    kategori|total_pendapatan|
+--------+----------+------+----------+------------+----------------+
|     TX0|        21|     1|      true|  Elektronik|           50000|
|     TX1|         8|     0|     false|     Fashion|         1500000|
|     TX2|        25|     2|      true|  Elektronik|          100000|
|     TX3|         3|     5|      true|     Fashion|          300000|
|     TX4|        19|     0|     false|Rumah Tangga|          450000|
|     TX5|        10|     5|      true|  Elektronik|          225000|
|     TX6|        26|     4|      true|   Kesehatan|          450000|
|     TX7|         4|     0|     false|     Makanan|          150000|
|     TX8|         7|     5|      true|  Elektronik|          150000|
|     TX9|        30|     0|     false|  Elektronik|          2500

---
## D. LOAD *(bobot 25%)*

**Penjelasan:** Hasil akhir pipeline disimpan ke HDFS dalam format **Parquet**, dipartisi berdasarkan `kategori` — mengikuti pola persis Sub-bab 6.4 (Tahap LOAD) pada modul. Path disesuaikan ke folder kerja pribadi (lihat catatan lingkungan di awal notebook).

In [16]:
!hdfs dfs -mkdir -p /home/mufid/praktikum-bigdata0046/tugas6

df_gabung.write.mode("overwrite").partitionBy("kategori").parquet(
    "hdfs://localhost:9000/home/mufid/praktikum-bigdata0046/tugas6/hasil_etl"
)

print("Pipeline ETL selesai — hasil tersimpan di HDFS.")
!hdfs dfs -ls -R /home/mufid/praktikum-bigdata0046/tugas6/hasil_etl

[Stage 37:>                                                         (0 + 1) / 1]

Pipeline ETL selesai — hasil tersimpan di HDFS.


-rw-r--r--   3 mufid supergroup          0 2026-09-30 18:18 /home/mufid/praktikum-bigdata0046/tugas6/hasil_etl/_SUCCESS
drwxr-xr-x   - mufid supergroup          0 2026-09-30 18:18 /home/mufid/praktikum-bigdata0046/tugas6/hasil_etl/kategori=Elektronik
-rw-r--r--   3 mufid supergroup      16366 2026-09-30 18:18 /home/mufid/praktikum-bigdata0046/tugas6/hasil_etl/kategori=Elektronik/part-00000-82635951-d586-4da1-a0ba-620f1aa79d77.c000.snappy.parquet
drwxr-xr-x   - mufid supergroup          0 2026-09-30 18:18 /home/mufid/praktikum-bigdata0046/tugas6/hasil_etl/kategori=Fashion
-rw-r--r--   3 mufid supergroup      11133 2026-09-30 18:18 /home/mufid/praktikum-bigdata0046/tugas6/hasil_etl/kategori=Fashion/part-00000-82635951-d586-4da1-a0ba-620f1aa79d77.c000.snappy.parquet
drwxr-xr-x   - mufid supergroup          0 2026-09-30 18:18 /home/mufid/praktikum-bigdata0046/tugas6/hasil_etl/kategori=Kesehatan
-rw-r--r--   3 mufid supergroup      10392 2026-09-30 18:18 /home/mufid/praktikum-bigdata0046/tu

In [17]:
# Verifikasi: membaca kembali dari HDFS dan menampilkan count() sebagai bukti data tersimpan utuh
df_final = spark.read.parquet("hdfs://localhost:9000/home/mufid/praktikum-bigdata0046/tugas6/hasil_etl")
print("Jumlah baris hasil akhir (dibaca kembali dari HDFS):", df_final.count())

Jumlah baris hasil akhir (dibaca kembali dari HDFS): 5000


---
## E. Insight Akhir *(bobot 15%)*

**Penjelasan:** Menggunakan DataFrame API, dihitung persentase transaksi dengan `ada_ulasan = True` untuk tiap kategori, lalu diurutkan dari yang **terendah** — kategori pada baris paling atas adalah jawabannya.

In [18]:
insight = df_final.groupBy("kategori").agg(
    count("order_id").alias("total_transaksi"),
    spark_round(
        count(when(col("ada_ulasan") == True, True)) / count("order_id") * 100, 2
    ).alias("persentase_ada_ulasan")
).orderBy(col("persentase_ada_ulasan").asc())

insight.show()

+------------+---------------+---------------------+
|    kategori|total_transaksi|persentase_ada_ulasan|
+------------+---------------+---------------------+
|     Makanan|            536|                68.84|
|   Kesehatan|            961|                68.89|
|     Fashion|           1035|                70.14|
|Rumah Tangga|            815|                70.55|
|  Elektronik|           1653|                70.66|
+------------+---------------+---------------------+



**Hasil:**

| kategori | total_transaksi | persentase_ada_ulasan |
|---|---|---|
| Makanan | 536 | **68,84%** |
| Kesehatan | 961 | 68,89% |
| Fashion | 1035 | 70,14% |
| Rumah Tangga | 815 | 70,55% |
| Elektronik | 1653 | 70,66% |

**Interpretasi bisnis:** Kategori **Makanan** memiliki persentase transaksi dengan ulasan **paling rendah** (68,84%), meskipun selisihnya dengan kategori lain relatif tipis (rentang keseluruhan hanya 68,84%-70,66%). Informasi ini penting bagi tim marketing karena rendahnya tingkat ulasan berarti platform kehilangan **sumber data sosial-proof** yang paling banyak dipakai calon pembeli untuk memutuskan pembelian — kategori dengan sedikit ulasan cenderung terlihat kurang terpercaya di mata pembeli baru, meskipun produknya sebenarnya laku. Tim marketing dapat menindaklanjuti temuan ini dengan mendorong kampanye *"berikan ulasan, dapatkan voucher"` khusus untuk kategori Makanan, atau mengirim pengingat otomatis pasca-pembelian yang lebih agresif untuk kategori-kategori dengan tingkat ulasan rendah, guna meningkatkan kepercayaan pembeli di masa mendatang.

---
## Menutup SparkSession

In [19]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
